# Foundations · Terraform

**Terraform** turns text files into cloud resources. You write the *end state* you want ("a container registry named X, a model deployment named Y"); Terraform compares it with what exists and makes the minimum changes.

Why we use it instead of clicking in the portal:

- **Repeatable.** The same files give every student the same platform, and give you the same platform after `destroy`.
- **Reviewable.** A change is a diff in a file, and `plan` shows its effect before anything happens.
- **Explainable.** The files *are* the documentation of what you run.

```mermaid
flowchart LR
    C["*.tf files<br/>(desired state)"] --> P["terraform plan<br/>(diff)"]
    S["terraform.tfstate<br/>(what Terraform created)"] --> P
    A["Azure<br/>(what exists)"] --> P
    P --> AP["terraform apply"] --> A
    AP --> S
```

This primer needs the resources from Setup. Nothing here changes them.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())))
import course

course.setup()

## 1 · A resource block

Everything in Terraform is a block. A `resource` block has a **type** (`azurerm_container_registry`, defined by the Azure provider), a **local name** (`this`, used to refer to it from other blocks) and **arguments**.

In [ ]:
print((course.INFRA / "registry.tf").read_text())

Read it line by line:

- `local.name` and `local.rg_name` are **locals**: values computed once in `main.tf`.
- `local.tags` adds the same tags to every resource, so you can find and bill them together.
- Other blocks refer to this one as `azurerm_container_registry.this.login_server`. That reference is also how Terraform knows the order: the Container App cannot be created before the registry it pulls from.

## 2 · Inputs, computed values, outputs

Variables are inputs (`variables.tf`), locals are derived values (`main.tf`), outputs are what you read back (`outputs.tf`). `terraform console` evaluates any expression against your real configuration.

In [ ]:
import shutil
import subprocess

for expression in ["var.student_id", "local.name", "local.foundry_project_endpoint", "var.chat_model"]:
    result = subprocess.run(
        [shutil.which("terraform"), "-chdir=infra", "console"], input=expression, capture_output=True, text=True
    )
    print(f"{expression:32s} = {result.stdout.strip()}")

## 3 · State: Terraform's memory

`terraform.tfstate` maps each block to the real Azure resource it created. Without it, Terraform would not know that `azurerm_container_registry.this` is the registry `acr…` in your subscription. Treat the state file like a password: it can contain secrets, so it is gitignored. Teams keep it in a shared, locked storage account instead (a *remote backend*).

In [ ]:
course.tf("state", "list")

In [ ]:
course.tf("state", "show", "azurerm_cognitive_deployment.chat")

## 4 · Change something, safely

You will add a tag to every resource, look at the plan, and **not** apply it. Any file ending in `.auto.tfvars` in `infra/` is read automatically.

In [ ]:
exercise = course.INFRA / "zz_exercise.auto.tfvars"
exercise.write_text('tags = { owner = "me", purpose = "terraform-primer" }\n')
try:
    course.tf("plan", "-input=false", "-no-color")
finally:
    exercise.unlink()
    print("Removed", exercise.name)

Read the plan:

- `~ update in-place` means Azure can change the resource without recreating it. Tags are always in place.
- `-/+ destroy and then create replacement` would mean the change forces a new resource, for example renaming it. That deletes data: always read the plan before you apply.
- The summary line counts adds, changes and destroys. Before every apply, make sure those numbers are what you expect.

## 5 · What to remember

- Files describe the end state; `plan` shows the diff; `apply` executes exactly the plan.
- References between blocks define the order. You rarely need `depends_on`.
- State is Terraform's memory. Keep it private; share it through a remote backend in teams.
- Never edit Terraform-managed resources in the portal: the next `plan` will try to undo your change.

**Learn more:** [Terraform on Azure (Microsoft Learn)](https://learn.microsoft.com/en-us/azure/developer/terraform/) · [Get started with Terraform on Azure (HashiCorp)](https://developer.hashicorp.com/terraform/tutorials/azure-get-started) · [azurerm provider docs](https://registry.terraform.io/providers/hashicorp/azurerm/latest/docs) · [Our infra README](../infra/README.md)